<a href="https://colab.research.google.com/github/bumbleboo021/AIC26_TayLor/blob/main/data_processing/extract_csv_keyframes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')
GDRIVE_DIR = "/content/drive/MyDrive/AIC26"
#mount vào drive batch 2, sửa csv_dir

Mounted at /content/drive


In [ ]:
!wget https://aic-data.ledo.io.vn/Videos_L21_a.zip
!unzip Videos_L21_a.zip #Cần chỉnh sau mỗi lần chạy

--2026-08-23 16:28:51--  https://aic-data.ledo.io.vn/Videos_L21_a.zip
Resolving aic-data.ledo.io.vn (aic-data.ledo.io.vn)... 104.21.75.130, 172.67.177.70, 2606:4700:3034::6815:4b82, ...
Connecting to aic-data.ledo.io.vn (aic-data.ledo.io.vn)|104.21.75.130|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3378949330 (3.1G) [application/zip]
Saving to: ‘Videos_L21_a.zip’

Videos_L21_a.zip    100%[===================>]   3.15G  84.0MB/s    in 46s     

2026-08-23 16:29:38 (69.5 MB/s) - ‘Videos_L21_a.zip’ saved [3378949330/3378949330]

Archive:  Videos_L21_a.zip
 extracting: video/L21_V001.mp4      
 extracting: video/L21_V002.mp4      
 extracting: video/L21_V003.mp4      
 extracting: video/L21_V005.mp4      
 extracting: video/L21_V006.mp4      
 extracting: video/L21_V007.mp4      
 extracting: video/L21_V008.mp4      
 extracting: video/L21_V009.mp4      
 extracting: video/L21_V010.mp4      
 extracting: video/L21_V011.mp4      
 extracting: video/L21_V012.mp4

In [ ]:
!pip install transnetv2_pytorch
import os
from transnetv2_pytorch import TransNetV2
import numpy as np
import pandas as pd
import subprocess
import gc
import torch
import glob
import cv2
import csv

In [ ]:
VIDEO_DIR = "/content/video"
CSV_DIR = "/content/mapkeyframes"
video_paths = glob.glob(os.path.join(VIDEO_DIR, "*.mp4"))
video_paths.sort()
model = TransNetV2()

In [ ]:
def extract_mapkeyframes_csv(video_path, csv_dir, model):
    os.makedirs(csv_dir, exist_ok=True)

    video_name = os.path.basename(video_path)
    video_id = os.path.splitext(video_name)[0]

    try:
        video_frames, single_preds, all_preds = model.predict_video(video_path)
        del video_frames, all_preds
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        gc.collect()

    except Exception as e:
        print(f"[{video_id}] Lỗi TransNet ở file {video_path}: {e}")

    scenes = model.predictions_to_scenes(single_preds.cpu().detach().numpy(), threshold = 0.5)
    del single_preds
    gc.collect()

    csv_filename = f"{video_id}.csv"
    csv_filepath = os.path.join(csv_dir, csv_filename)
    with open(csv_filepath, mode='w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(["n_keyframes", "frame_idx"])
        n_keyframes = 1

        for scene in scenes:
            start, end = scene
            length = end - start
            frames_to_add = []
            if length < 500:
                frames_to_add = [
                    start,
                    start + length // 2,
                    end
                ]
            elif length < 1000:
                frames_to_add = [
                    start,
                    start + length // 3,
                    start + 2 * (length) // 3,
                    end
                ]
            else:
                frames_to_add = [
                    start,
                    start + length // 4,
                    start + 2 * (length) // 4,
                    start + 3 * (length) // 4,
                    end
                ]
            for frame in frames_to_add:
                writer.writerow([n_keyframes, frame])
                n_keyframes += 1

    print(f"[{video_id}] Done")

In [ ]:
for video_path in video_paths:
    video_filename = os.path.basename(video_path)
    video_id, _ = os.path.splitext(video_filename)
    lxx_prefix = video_id.split('_')[0]
    try:
        extract_mapkeyframes_csv(video_path, CSV_DIR, model)
    except Exception as e:
        print(f"Lỗi khi xử lý {video_id}: {e}\n")

Extracting frames from /content/video/L21_V001.mp4


Processing frames: 100%|██████████| 37849/37849 [00:32<00:00, 1153.30frame/s]


[L21_V001] Done
Extracting frames from /content/video/L21_V002.mp4


Processing frames: 100%|██████████| 31720/31720 [00:27<00:00, 1140.52frame/s]


[L21_V002] Done
Extracting frames from /content/video/L21_V003.mp4


...

[L21_V029] Done
Extracting frames from /content/video/L21_V030.mp4


Processing frames: 100%|██████████| 31801/31801 [00:28<00:00, 1105.92frame/s]


[L21_V030] Done
Extracting frames from /content/video/L21_V031.mp4


Processing frames: 100%|██████████| 27528/27528 [00:24<00:00, 1116.32frame/s]


[L21_V031] Done


In [ ]:
!cd mapkeyframes && zip -r ../mapkeyframes .

  adding: L21_V022.csv (deflated 56%)
  adding: L21_V012.csv (deflated 56%)
  adding: L21_V010.csv (deflated 56%)
  adding: L21_V021.csv (deflated 56%)
  adding: L21_V018.csv (deflated 56%)
  adding: L21_V014.csv (deflated 56%)
  adding: L21_V005.csv (deflated 56%)
  adding: L21_V011.csv (deflated 56%)
  adding: L21_V026.csv (deflated 56%)
  adding: L21_V002.csv (deflated 56%)
  adding: L21_V006.csv (deflated 56%)
  adding: L21_V031.csv (deflated 56%)
  adding: L21_V015.csv (deflated 56%)
  adding: L21_V030.csv (deflated 56%)
  adding: L21_V008.csv (deflated 56%)
  adding: L21_V027.csv (deflated 56%)
  adding: L21_V024.csv (deflated 56%)
  adding: L21_V007.csv (deflated 56%)
  adding: L21_V001.csv (deflated 56%)
  adding: L21_V019.csv (deflated 56%)
  adding: L21_V029.csv (deflated 56%)
  adding: L21_V023.csv (deflated 56%)
  adding: L21_V013.csv (deflated 56%)
  adding: L21_V028.csv (deflated 56%)
  adding: L21_V017.csv (deflated 55%)
  adding: L21_V009.csv (deflated 56%)
  adding: L2